# Phase 98: COACH420 ESM2-3B: SCORE-ONLY

Second half of the Phase93 split (see Phase97). Loads Phase97's 3 trained
checkpoints fresh (rebuilding `LocalizingEGNN` from the same N_PRE=2587/
TOPO_DIM=34 the checkpoints were trained with) into a kernel that has
NEVER held the 7,513-chain training pool -- so the live ESM2-3B load for
featurizing the 420 benchmark structures has the full RAM budget to
itself, unlike Phase93 v2-v6 which all crashed doing this in the same
process as training.

## Cell 0: Install dependencies (a fresh kernel needs this too -- v1 of this notebook forgot it entirely, ModuleNotFoundError on torch_geometric)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3 accelerate
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1

## Cell 1: Lightweight setup: Phase51's small topology blob only (NOT Phase96's big training blob), hardcoded N_PRE/TOPO_DIM/ESM_DIM (confirmed via Phase96/97's own log: node dim 2621 = 21+6+2560+34)

In [ ]:
import csv as _csv
import gzip, json, os, pickle, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from Bio.PDB import PDBParser
from scipy.spatial import cKDTree
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob6b = torch.load(locate("phase51_graphs.pt"), map_location="cpu", weights_only=False)
TOPO_NAMES = blob6b["topo_names"]
TOPO_DIM = len(TOPO_NAMES)
assert TOPO_DIM == 34, f"expected TOPO_DIM=34 (matches Phase97's trained checkpoints), got {TOPO_DIM}"
N_PRE = 2587    # 21 onehot + 6 dihedral + 2560 ESM -- must match what Phase97 trained with
ESM_DIM = 2560
if "dataset" in blob6b: del blob6b["dataset"]   # only topo_mean/topo_std/topo_names needed later

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"N_PRE={N_PRE}  TOPO_DIM={TOPO_DIM}  ESM_DIM={ESM_DIM}  DEVICE={DEVICE}")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

## Cell 2: Fetch ALL 420 COACH420 structures (identical to Phase 20/31/38/53/93/97)

In [ ]:
def parse_ds(url):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        out.append((name[:4].upper(), name[4:].upper() or None))
    return out

coach = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/coach420.ds")
print(f"COACH420: {len(coach)} entries (full benchmark, no filtering)")

need = sorted({pid for pid, _ in coach})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
coach_paths = {p: f for p, f in got if f}
print(f"fetched {len(coach_paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 3: Model (identical to every prior localization phase -- needed here to reconstruct the trained architecture for loading checkpoints)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Load Phase97's 3 trained checkpoints into fresh model instances

In [ ]:
ensemble = []
for seed in (0, 1, 2):
    m = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=True).to(DEVICE)
    ckpt = locate(f"clean_deploy_esm3b_seed{seed}.pt")
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    ensemble.append(m)
    print(f"  loaded seed {seed} from {ckpt}")
print(f"ensemble of {len(ensemble)} trained models ready")

## Cell 5: Ground-truth ligands for ALL 420 COACH420 structures (identical to Phase 20/31/38/53/93)

In [ ]:
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_RES, MAX_RES, MIN_LIG_ATOMS = 50, 1000, 6

def parse_structure(pid, path, wanted_chain):
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
    except Exception:
        return []
    ligs = []
    for ch in model:
        for res in ch:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                            dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    out = []
    chains = [model[wanted_chain]] if (wanted_chain and wanted_chain in [c.id for c in model]) \
             else [c for c in model]
    for ch in chains:
        residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not (MIN_RES <= len(residues) <= MAX_RES): continue
        ca=[r["CA"].coord for r in residues]
        seq="".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
        atoms=[np.asarray([a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")],
                          dtype=np.float32).reshape(-1,3) for r in residues]
        out.append((ch.id, seq, np.asarray(ca,dtype=np.float32), atoms, ligs))
    return out

t0=time.time()
eval_units = []
for pid, ch in coach:
    if pid not in coach_paths: continue
    chains = parse_structure(pid, coach_paths[pid], ch)
    if chains: eval_units.append((pid, chains[:1]))   # coach420: one specific target chain
print(f"evaluable COACH420 structures: {len(eval_units)}/{len(coach)}  ({(time.time()-t0)/60:.1f} min)")

## Cell 6: Featurize (dihedral + ESM2-3B + topology; ESM2-3B loaded fresh here, full RAM budget available since training never ran in this process)

In [ ]:
import gudhi
from transformers import AutoTokenizer, AutoModel
from Bio.PDB import PPBuilder

AA_ALPHABET="ACDEFGHIKLMNPQRSTVWYX"; A2I={a:i for i,a in enumerate(AA_ALPHABET)}
MAX_HOM_DIM=2; BETTI_RADII=(2.0,3.0,5.0); NBHD_RADIUS=12.0; MAX_NBHD_ATOMS=300
FPD=6+2+len(BETTI_RADII)

_FULL_TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM+1)
                    for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                              "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
                   + ["nbhd_n_atoms","nbhd_gyration"]
_TOPO_KEEP = np.array([n in set(TOPO_NAMES) for n in _FULL_TOPO_NAMES])
assert _TOPO_KEEP.sum() == TOPO_DIM

def rgc(pos,r=8.0,k=32):
    n=pos.size(0); pos=pos-pos.mean(0,keepdim=True)
    d=torch.cdist(pos,pos).clone(); d.fill_diagonal_(float("inf")); m=d<=r
    if k<n:
        _,i=torch.topk(d,k,dim=1,largest=False); keep=torch.zeros_like(m); keep.scatter_(1,i,True); m&=keep
    m=m|m.t(); t,s=m.nonzero(as_tuple=True); return torch.stack([s,t]).long()

def _dg(pts):
    pts=np.ascontiguousarray(pts,dtype=np.float64); st=None; sq=True
    if len(pts)>=5:
        try: st=gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception: st=None
    if st is None:
        st=gudhi.RipsComplex(points=pts,max_edge_length=float(NBHD_RADIUS)).create_simplex_tree(max_dimension=MAX_HOM_DIM+1); sq=False
    st.compute_persistence(homology_coeff_field=2,persistence_dim_max=True)
    o={}
    for d in range(MAX_HOM_DIM+1):
        b=np.asarray(st.persistence_intervals_in_dimension(d),dtype=np.float64).reshape(-1,2)
        o[d]=np.sqrt(np.maximum(b,0.0)) if (sq and len(b)) else b
    return o

def _vec(dg,pts):
    f=[]
    for d in range(MAX_HOM_DIM+1):
        bars=dg.get(d,np.zeros((0,2)))
        fin=bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1,2)
        if len(fin)==0: f.extend([0.0]*FPD); continue
        b,dt=fin[:,0],fin[:,1]; pers=np.maximum(dt-b,0.0); tot=float(pers.sum())
        ent=0.0
        if tot>0:
            p=pers/tot; nz=p[p>0]; ent=float(-(nz*np.log(nz)).sum())
        f.extend([float(len(fin)),tot,float(pers.max()),float(pers.mean()),float(pers.std()),
                  ent,float(b.mean()),float(dt.mean()),
                  *[float(((b<=r)&(dt>r)).sum()) for r in BETTI_RADII]])
    c=pts-pts.mean(0,keepdims=True)
    f.extend([float(len(pts)),float(np.sqrt((c**2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(f,dtype=np.float32)

def topo_chain(ca,per_res):
    atoms=np.concatenate([a for a in per_res if len(a)],axis=0)
    tree=cKDTree(atoms); out=np.zeros((len(ca),TOPO_DIM),np.float32)
    for i,c in enumerate(ca):
        idx=tree.query_ball_point(c,NBHD_RADIUS)
        if len(idx)>MAX_NBHD_ATOMS: _,idx=tree.query(c,k=MAX_NBHD_ATOMS)
        pts=atoms[np.asarray(idx,dtype=int)]
        if len(pts)>=4: out[i]=_vec(_dg(pts),pts)[_TOPO_KEEP]
    return out

def dihedrals(pdb_id, chain_id, path, n_expect):
    try:
        with gzip.open(path,"rt") as fh:
            model=next(PDBParser(QUIET=True,PERMISSIVE=True).get_structure(pdb_id,fh).get_models())
        ch=model[chain_id]
    except Exception: return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues)!=n_expect: return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()): ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi=ppmap.get(r.id,(None,None)); phis.append(phi); psis.append(psi)
    def enc(a):
        v=torch.tensor([x if x is not None else 0.0 for x in a],dtype=torch.float32)
        m=torch.tensor([x is not None for x in a],dtype=torch.float32)
        return torch.stack([torch.sin(v)*m,torch.cos(v)*m],1), m.unsqueeze(1)
    pf,pm=enc(phis); sf,sm=enc(psis)
    return torch.cat([pf,sf,pm,sm],1)

tok = AutoTokenizer.from_pretrained("facebook/esm2_t36_3B_UR50D")
esm = (AutoModel.from_pretrained("facebook/esm2_t36_3B_UR50D", torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
       if DEVICE.type == "cuda" else
       AutoModel.from_pretrained("facebook/esm2_t36_3B_UR50D")).eval()  # low_cpu_mem_usage+.to() leaves a meta tensor on CPU-only runs (NotImplementedError); only use it for GPU

@torch.no_grad()
def embed_seq(seq):
    enc = tok([seq], return_tensors="pt", truncation=True, max_length=1002)
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    out = esm(**enc).last_hidden_state
    return out[0,1:1+len(seq),:].float().cpu()  # 3B model is fp16; upcast here

t0=time.time(); built = {}
n_total = sum(len(c) for _,c in eval_units)
done=0
for pid, chains in eval_units:
    for cid, seq, ca, per_res, ligs in chains:
        n = len(ca)
        oh = torch.zeros(n, 21); oh[torch.arange(n), torch.tensor([A2I.get(c,20) for c in seq])]=1.0
        dih = dihedrals(pid, cid, coach_paths[pid], n)
        if dih is None: continue
        esm_e = embed_seq(seq)
        topo = topo_chain(ca, per_res)
        pos = torch.from_numpy(ca); pos = pos - pos.mean(0, keepdim=True)
        x = torch.cat([oh, dih, esm_e, torch.from_numpy(topo.astype(np.float32))], dim=1)
        ei = rgc(pos)
        d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
        sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)
        built[(pid,cid)] = dict(x=x, pos=pos, edge_index=ei,
                                edge_attr=torch.cat([d_,torch.log1p(sep)],1), ca=ca, ligs=ligs)
        done += 1
    if done % 40 < 1: print(f"  {done}/{n_total}  [{(time.time()-t0)/60:.1f}m]")

del esm; torch.cuda.empty_cache()
print(f"\nfeaturized {len(built)}/{n_total} chains in {(time.time()-t0)/60:.1f} min")

raw_cache = {k: {kk: (vv.numpy() if torch.is_tensor(vv) else vv) for kk,vv in v.items()}
            for k,v in built.items()}
with open(WORK/"cache/phase98_raw_features.pkl", "wb") as fh:
    pickle.dump(raw_cache, fh)
print(f"cached raw features -> {(WORK/'cache/phase98_raw_features.pkl').stat().st_size/1e6:.0f} MB")

## Cell 7: P2Rank on the FULL 420 (same-run controlled reference, identical to Phase 20/31/38/53/93)

In [ ]:
t0=time.time(); PRANK=None
subprocess.run("curl -sL https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz "
               "-o /tmp/p2.tar.gz && tar xzf /tmp/p2.tar.gz -C /tmp", shell=True, check=True, timeout=1200)
PRANK = str(next(Path("/tmp").glob("p2rank_*/prank"))); os.chmod(PRANK, 0o755)

pW = Path("/tmp/p2_98"); shutil.rmtree(pW, ignore_errors=True); pW.mkdir()
pOUT = Path("/tmp/p2_98_out"); shutil.rmtree(pOUT, ignore_errors=True)
for pid in sorted(coach_paths):
    plain = pW / f"{pid}.pdb"
    if not plain.exists():
        with gzip.open(coach_paths[pid],"rb") as fi, open(plain,"wb") as fo: shutil.copyfileobj(fi,fo)
(pW/"ds.ds").write_text("\n".join(str(pW/f"{p}.pdb") for p in sorted(coach_paths))+"\n")

r = subprocess.run([PRANK,"predict",str(pW/"ds.ds"),"-o",str(pOUT),
                    "-threads", str(max(1, os.cpu_count() or 2))],
                   capture_output=True, text=True, timeout=3600)
print(f"P2Rank rc={r.returncode} in {(time.time()-t0)/60:.1f} min")
pred_csvs = {p.stem.replace(".pdb_predictions",""): p for p in pOUT.rglob("*_predictions.csv")}
print(f"P2Rank outputs: {len(pred_csvs)}/{len(coach_paths)}")

def p2rank_top_center(pid):
    f = pred_csvs.get(pid)
    if f is None: return None
    with open(f) as fh:
        rows = list(_csv.DictReader(fh))
    if not rows: return None
    rows = [{k.strip(): v.strip() if isinstance(v,str) else v for k,v in row.items() if k} for row in rows]
    cx = next((k for k in rows[0] if k.lower().replace("_","")=="centerx"), None)
    cy = next((k for k in rows[0] if k.lower().replace("_","")=="centery"), None)
    cz = next((k for k in rows[0] if k.lower().replace("_","")=="centerz"), None)
    if not (cx and cy and cz): return None
    try:
        return np.array([float(rows[0][cx]), float(rows[0][cy]), float(rows[0][cz])], dtype=np.float32)
    except (ValueError, KeyError):
        return None

p2rank_centers = {pid: p2rank_top_center(pid) for pid in coach_paths}
print(f"P2Rank centers parsed: {sum(1 for v in p2rank_centers.values() if v is not None)}/{len(coach_paths)}")

## Cell 8: Score both, compare to literature AND to the 35M-vs-3B comparison

In [ ]:
mu, sd = blob6b["topo_mean"], blob6b["topo_std"]

@torch.no_grad()
def predict_chain(entry):
    x = entry["x"].clone()
    x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu)) / torch.from_numpy(sd)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=entry["pos"].to(DEVICE)
    d.edge_index=entry["edge_index"].to(DEVICE); d.edge_attr=entry["edge_attr"].to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    seed_logits, seed_centers = [], []
    for m in ensemble:
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])
    offset = entry["ca"].mean(axis=0)
    center_raw = np.mean(seed_centers, axis=0) + offset
    return center_raw


def dcc_dca(pred_center, ligs):
    if pred_center is None or not ligs: return None, None
    best_dcc, best_dca = np.inf, np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        com = lig_atoms.mean(axis=0)
        d_ = float(np.linalg.norm(pred_center - com))
        if d_ < best_dcc:
            best_dcc = d_
            best_dca = float(np.linalg.norm(lig_atoms - pred_center, axis=1).min())
    return (best_dcc if np.isfinite(best_dcc) else None, best_dca if np.isfinite(best_dca) else None)


ours_dcc, ours_dca, p2_dcc, p2_dca = [], [], [], []
for (pid, cid), entry in built.items():
    ligs = entry["ligs"]
    c_ours = predict_chain(entry)
    d1, a1 = dcc_dca(c_ours, ligs)
    if d1 is not None: ours_dcc.append(d1); ours_dca.append(a1)
    c_p2 = p2rank_centers.get(pid)
    if c_p2 is not None:
        d2, a2 = dcc_dca(c_p2, ligs)
        if d2 is not None: p2_dcc.append(d2); p2_dca.append(a2)

ours_dcc_rate = float(np.mean([1.0 if d<4.0 else 0.0 for d in ours_dcc]))
ours_dca_rate = float(np.mean([1.0 if a<4.0 else 0.0 for a in ours_dca]))
p2_dcc_rate = float(np.mean([1.0 if d<4.0 else 0.0 for d in p2_dcc])) if p2_dcc else float("nan")
p2_dca_rate = float(np.mean([1.0 if a<4.0 else 0.0 for a in p2_dca])) if p2_dca else float("nan")

LIT = {"fpocket": (0.228, 0.444), "P2Rank": (0.366, 0.628), "EquiPocket": (0.423, 0.656)}

print(f"{'method':<28}{'DCC':>8}{'DCA':>8}{'n':>6}")
print("-"*50)
for name,(d,a) in LIT.items():
    print(f"{name+' (literature, full 420)':<28}{d:>8.3f}{a:>8.3f}{'420':>6}")
print(f"{'P2Rank (this run, full 420)':<28}{p2_dcc_rate:>8.3f}{p2_dca_rate:>8.3f}{len(p2_dcc):>6}")
print(f"{'OURS (clean train, full 420)':<28}{ours_dcc_rate:>8.3f}{ours_dca_rate:>8.3f}{len(ours_dcc):>6}")

print(f"\nvs literature EquiPocket : DCC {ours_dcc_rate-0.423:+.4f}  DCA {ours_dca_rate-0.656:+.4f}")
print(f"vs literature P2Rank     : DCC {ours_dcc_rate-0.366:+.4f}  DCA {ours_dca_rate-0.628:+.4f}")
print(f"vs this-run P2Rank       : DCC {ours_dcc_rate-p2_dcc_rate:+.4f}  DCA {ours_dca_rate-p2_dca_rate:+.4f}")

verdict = ("BEATS literature EquiPocket on the full, unfiltered benchmark -- "
          "a legitimate SOTA claim" if ours_dcc_rate > 0.423 else
          "beats P2Rank but not EquiPocket on the full benchmark" if ours_dcc_rate > 0.366 else
          "does not beat published baselines on the full, unfiltered benchmark")
print(f"\nVERDICT: {verdict}")

json.dump({"ours": {"dcc": ours_dcc_rate, "dca": ours_dca_rate, "n": len(ours_dcc)},
           "p2rank_this_run": {"dcc": p2_dcc_rate, "dca": p2_dca_rate, "n": len(p2_dcc)},
           "literature": LIT, "n_train_excluded": 332, "n_train_clean": 7181},  # from Phase97's own log
          open(WORK/"phase98_summary.json", "w"), indent=2)
print("\nsaved -> phase98_summary.json")

In [ ]:
print(f"\n--- COACH420 DCC: ESM2 capacity lever at round-3 scale ---")
print(f"{'embedding':<28}{'hidden':>8}{'DCC (3-seed)':>14}{'DCC (8-seed)':>14}")
print("-"*64)
print(f"{'ESM2-35M (Phase53/56)':<28}{480:>8}{0.5763:>14.4f}{0.5932:>14.4f}")
print(f"{'ESM2-3B (this notebook)':<28}{2560:>8}{ours_dcc_rate:>14.4f}{'pending':>14}")
print(f"\ntraining pool size: 7,181 clean chains (round-3, unchanged, from Phase97)")
print(f"this-run P2Rank DCC: {p2_dcc_rate:.4f}  (deterministic tool, should match prior rounds' ~0.5209)")

---

## Reading this

- **DCC beats 0.5932** -- embedding capacity is a real, addressable lever
  here too (mirrors CryptoBench), worth extending to 8 seeds and then
  applying to HOLO4K (Phase99/100).
- **DCC roughly matches 0.5932 (within noise)** -- capacity doesn't help
  once topology+dihedral+scale have already saturated what COACH420's
  training signal can support.
- **DCC drops** -- would be surprising and worth real scrutiny before
  accepting.